In [1]:
# ===== [KAGGLE] bootstrap: recreate the Colab layout =====
# The paper notebooks expect /content/drive/MyDrive/bsm_paper_replication and
# /content/BSM_Project_2. bsm_env builds both from the attached dataset; the
# pipeline itself (images, augmentation, splits, training) is unchanged.
import pathlib, sys
_code = pathlib.Path("/kaggle/input/bsm-paper-2-kaggle/code")
if not _code.is_dir():
    _hits = list(pathlib.Path("/kaggle/input").rglob("bsm_env.py"))
    _code = _hits[0].parent if _hits else _code
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import bsm_env
BSM_RAW_SRC = bsm_env.setup()


[bsm_env] using existing /content
[bsm_env] mode=existing  project=/content/drive/MyDrive/bsm_paper_replication
[bsm_env] raw images=367  originals=367
[bsm_env] models.py: ok
[bsm_env] common.py: ok
[bsm_env] torch 2.11.0+cu128 cuda=True Tesla T4
[bsm_env] disk /kaggle/working free=20.9 GB


# BSM Paper Replication — Notebook 01: EDA

**Paper stage:** Data pre-processing layer — Exploratory Data Analysis
(*"EDA offers statistical perspectives on the dataset"*)

Pipeline position (paper Fig. 1): `BSM data → EDA → Data augmentation → Data imputation → Data wrangling → Data versioning`

This notebook:
1. Sets up the environment (Drive, seed 42, library versions)
2. Downloads the Kaggle BSM dataset (367 images, 6 mineral classes)
3. Builds and saves the raw inventory
4. Class counts vs the paper's Fig. 2 (and its 357 / 367 discrepancy)
5. Image statistics (size, channels, format, brightness per class)
6. Reproduces paper **Fig. 2** (class-count bar chart) and **Fig. 3** (image grid)
7. Writes the shared modules `models.py` and `common.py` to Drive

*Faithful replication — see DEVIATIONS.md in the repo (holisticNoobda/BSM-Project-2).*

## Common setup

Same first cell in every notebook: mount Drive, create the project folders, seed 42, print library versions and GPU.

In [2]:
# ===== COMMON SETUP CELL (same in every notebook, spec section 4) =====
import os, sys, random, time
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# --- Google Drive (persistent storage; click "Allow" on the popup) ---
# [kaggle] Colab Drive mount removed - bsm_env recreates the folder tree

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
for sub in ["data", "augmented", "arrays", "checkpoints", "results", "figures", "versions"]:
    os.makedirs(os.path.join(PROJECT, sub), exist_ok=True)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import torch
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("project root :", PROJECT)
print("python       :", sys.version.split()[0])
print("numpy        :", np.__version__)
print("torch        :", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("GPU          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("torchvision  :", end=" ")
try:
    import torchvision; print(torchvision.__version__)
except Exception:
    print("not installed (not needed — models are built from the paper's tables)")
t0 = time.time(); _ = [i*i for i in range(10**6)]; print("setup OK  (%.2fs)" % (time.time()-t0))

project root : /content/drive/MyDrive/bsm_paper_replication
python       : 3.13.15
numpy        : 2.1.3
torch        : 2.11.0+cu128
cuda available: True
GPU          : Tesla T4
torchvision  : 0.26.0+cu128
setup OK  (0.08s)


## Paper stage: dataset acquisition (Kaggle, via Colab Secrets)

In [3]:
# ===== [KAGGLE] dataset acquisition =====
# On Colab this cell downloaded the public dataset [15] with kagglehub using Colab
# Secrets. On Kaggle the same 367 originals are attached read-only, so the
# acquisition step reduces to inspecting the mount.
import pathlib
from collections import Counter

CACHE = pathlib.Path(BSM_RAW_SRC)
files = [f for f in CACHE.rglob("*") if f.is_file()]
dirs = sorted({f.parent.name for f in files})
print("dataset mount:", CACHE)
print("top-level folders:", dirs)
print("total files:", len(files))
print("files per folder:", dict(Counter(f.parent.name for f in files)))


dataset mount: /content/BSM_Project_2/data/raw/BSM
top-level folders: ['Garnet', 'Ilmenite', 'Monazite', 'Rutile', 'Sillimanite', 'Zircon']
total files: 367
files per folder: {'Sillimanite': 60, 'Monazite': 60, 'Rutile': 66, 'Garnet': 60, 'Zircon': 59, 'Ilmenite': 62}


## Paper stage: raw inventory (367 images, layout inspected not assumed)

In [4]:
# ===== Paper stage: Data pre-processing — raw inventory =====
# Layout is INSPECTED, not assumed (spec section 3): find the class folders as
# the directories that directly contain image files.
import os, pathlib
import pandas as pd
from PIL import Image

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
CACHE = pathlib.Path(BSM_RAW_SRC)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
all_files = [f for f in CACHE.rglob("*") if f.is_file()]
img_files = [f for f in all_files if f.suffix.lower() in IMG_EXT]
class_dirs = sorted({f.parent for f in img_files})
print("discovered layout:")
print("  total files :", len(all_files), "| image files:", len(img_files), "| non-image:", len(all_files) - len(img_files))
print("  class folders:", [d.name for d in class_dirs])

# Paper Fig. 2 / confusion-matrix class order (spec section 1).
CLASS_ORDER = ["Ilmenite", "Rutile", "Zircon", "Garnet", "Monazite", "Sillimanite"]
assert sorted(d.name for d in class_dirs) == sorted(CLASS_ORDER), "unexpected class folders!"
name_to_id = {n: i for i, n in enumerate(CLASS_ORDER)}

rows = []
for d in class_dirs:
    for f in sorted(d.iterdir()):
        if not f.is_file() or f.suffix.lower() not in IMG_EXT:
            continue
        rec = {"image_id": f.stem, "filepath": str(f), "class_name": d.name,
               "class_id": name_to_id[d.name], "file_size_bytes": f.stat().st_size,
               "extension": f.suffix.lower()}
        try:
            with Image.open(f) as im:
                im.verify()
            with Image.open(f) as im:
                w, h = im.size
                rec.update({"width": w, "height": h, "channels": len(im.getbands()),
                            "mode": im.mode, "format": im.format, "readable": True})
        except Exception as e:
            rec.update({"width": None, "height": None, "channels": None,
                        "mode": None, "format": None, "readable": False})
            print("UNREADABLE:", f.name, type(e).__name__)
        rows.append(rec)

inv = pd.DataFrame(rows).sort_values(["class_id", "image_id"]).reset_index(drop=True)
inv_path = os.path.join(PROJECT, "results", "raw_inventory.csv")
inv.to_csv(inv_path, index=False)
print("\ninventory rows:", len(inv), "-> saved", inv_path)
print("unreadable:", int((~inv["readable"]).sum()))
print(inv.head(3).to_string(index=False))
print()
inv[["width", "height", "channels", "file_size_bytes"]].describe().round(1).to_string()

discovered layout:
  total files : 367 | image files: 367 | non-image: 0
  class folders: ['Garnet', 'Ilmenite', 'Monazite', 'Rutile', 'Sillimanite', 'Zircon']

inventory rows: 367 -> saved /content/drive/MyDrive/bsm_paper_replication/results/raw_inventory.csv
unreadable: 0
image_id                                            filepath class_name  class_id  file_size_bytes extension  width  height  channels mode format  readable
       1  /content/BSM_Project_2/data/raw/BSM/Ilmenite/1.jpg   Ilmenite         0            11605      .jpg    241     172         3  RGB   JPEG      True
      10 /content/BSM_Project_2/data/raw/BSM/Ilmenite/10.jpg   Ilmenite         0            16401      .jpg    268     198         3  RGB   JPEG      True
      11 /content/BSM_Project_2/data/raw/BSM/Ilmenite/11.jpg   Ilmenite         0            10071      .jpg    184     183         3  RGB   JPEG      True



'       width  height  channels  file_size_bytes\ncount  367.0   367.0     367.0            367.0\nmean   277.7   218.5       3.0          19255.2\nstd     95.7    69.7       0.0          15347.4\nmin    102.0    91.0       3.0           3020.0\n25%    215.5   173.5       3.0          10352.0\n50%    253.0   192.0       3.0          14588.0\n75%    323.0   251.0       3.0          22895.5\nmax    765.0   641.0       3.0         168769.0'

## Paper stage: class counts and per-class statistics vs paper Fig. 2

In [5]:
# ===== Paper stage: Data pre-processing — class counts vs paper Fig. 2 =====
import pandas as pd

inv = pd.read_csv("/content/drive/MyDrive/bsm_paper_replication/results/raw_inventory.csv")
CLASS_ORDER = ["Ilmenite", "Rutile", "Zircon", "Garnet", "Monazite", "Sillimanite"]

# Paper Fig. 2 counts (spec section 1).
paper_fig2 = {"Ilmenite": 60, "Rutile": 62, "Zircon": 60, "Garnet": 66, "Monazite": 60, "Sillimanite": 59}

counts = inv["class_name"].value_counts()
summary = pd.DataFrame({
    "class": CLASS_ORDER,
    "real_count": [int(counts.get(c, 0)) for c in CLASS_ORDER],
    "paper_Fig2": [paper_fig2[c] for c in CLASS_ORDER],
})
summary["match"] = summary["real_count"] == summary["paper_Fig2"]
summary["percent"] = (summary["real_count"] / summary["real_count"].sum() * 100).round(2)
print(summary.to_string(index=False))

print()
real_total = int(summary["real_count"].sum())
fig2_sum = int(summary["paper_Fig2"].sum())
print(f"REAL downloaded total      : {real_total}")
print(f"paper Fig. 2 sum           : {fig2_sum}  (matches real: {fig2_sum == real_total})")
print(f"paper TEXT total ('357')   : 357  (matches real: {357 == real_total})")
print("=> The paper's text total 357 contradicts its own Fig. 2 (sums to 367).")
print("   We report all three numbers; the real count governs (DEVIATIONS row 1).")

# Mean brightness per class (grayscale mean of each image).
import numpy as np
from PIL import Image
bright = []
for name, grp in inv.groupby("class_name"):
    vals = []
    for fp in grp["filepath"]:
        with Image.open(fp) as im:
            vals.append(np.asarray(im.convert("L"), dtype=np.float32).mean())
    bright.append({"class": name, "mean_brightness": round(float(np.mean(vals)), 2),
                   "std_brightness": round(float(np.std(vals)), 2)})
br = pd.DataFrame(bright).set_index("class").loc[CLASS_ORDER]
print()
print("mean brightness per class (0-255 grayscale):")
print(br.to_string())

      class  real_count  paper_Fig2  match  percent
   Ilmenite          62          60  False    16.89
     Rutile          66          62  False    17.98
     Zircon          59          60  False    16.08
     Garnet          60          66  False    16.35
   Monazite          60          60   True    16.35
Sillimanite          60          59  False    16.35

REAL downloaded total      : 367
paper Fig. 2 sum           : 367  (matches real: True)
paper TEXT total ('357')   : 357  (matches real: False)
=> The paper's text total 357 contradicts its own Fig. 2 (sums to 367).
   We report all three numbers; the real count governs (DEVIATIONS row 1).

mean brightness per class (0-255 grayscale):
             mean_brightness  std_brightness
class                                       
Ilmenite              158.96           43.02
Rutile                166.64           47.64
Zircon                150.26           49.53
Garnet                130.91           41.31
Monazite              118.34

## Paper stage: figures — paper Fig. 2 (class counts) and Fig. 3 (image grid) reproduced

In [6]:
# ===== Paper stage: Data pre-processing — reproduce paper Fig. 2 =====
# "Bar chart of class counts (title-less, 'count' on y, class names rotated,
#  one colour per class)" — spec section 5.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

inv = pd.read_csv("/content/drive/MyDrive/bsm_paper_replication/results/raw_inventory.csv")
CLASS_ORDER = ["Ilmenite", "Rutile", "Zircon", "Garnet", "Monazite", "Sillimanite"]
counts = inv["class_name"].value_counts().reindex(CLASS_ORDER)

colors = plt.cm.tab10.colors[:6]
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(counts.index, counts.values, color=colors)
ax.set_ylabel("count")
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
fig.savefig("/content/drive/MyDrive/bsm_paper_replication/figures/fig02_class_counts.png", dpi=150)
plt.show()

# ===== Paper Fig. 3: large grid of images covering the six classes =====
# "about 8 rows x 15 columns, each image shown at 224x224" — spec section 5.
import numpy as np
from PIL import Image

inv_s = inv.sample(frac=1.0, random_state=42).reset_index(drop=True)  # shuffled, seed 42
NROWS, NCOLS = 8, 15
n = NROWS * NCOLS  # 120
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(3 * NCOLS, 3 * NROWS))
for i, ax in enumerate(axes.ravel()):
    if i >= n:
        ax.axis("off"); continue
    with Image.open(inv_s.loc[i, "filepath"]) as im:
        ax.imshow(im.convert("RGB").resize((224, 224)))
    ax.set_title(inv_s.loc[i, "class_name"], fontsize=8)
    ax.axis("off")
fig.suptitle("")
fig.tight_layout()
fig.savefig("/content/drive/MyDrive/bsm_paper_replication/figures/fig03_image_grid.png", dpi=100)
plt.show()
print("Fig. 2 and Fig. 3 saved to Drive /bsm_paper_replication/figures/")

Fig. 2 and Fig. 3 saved to Drive /bsm_paper_replication/figures/


## Paper stage: shared modules (`models.py`, `common.py`) written to Drive

Canonical source: the public repo `holisticNoobda/BSM-Project-2`. The SHA-256 checksums are printed so the exact verified code is auditable.

In [7]:
# ===== Paper stage: write shared modules to Drive (spec section 4) =====
# Canonical source: this repo (public). The local verified copies are pushed
# there; we pull them to the Drive project folder so every notebook imports
# the SAME verified code. Checksums printed for auditability.
import hashlib, os, urllib.request

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
REPO_RAW = "https://raw.githubusercontent.com/holisticNoobda/BSM-Project-2/main"

for fname in ["models.py", "common.py"]:
    dst = os.path.join(PROJECT, fname)
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/{fname}", dst)
    except Exception as _dl_err:
        print("  [kaggle] download failed, using local copy:", _dl_err)
        assert os.path.exists(dst), f"{fname} missing and no network"
    digest = hashlib.sha256(open(dst, "rb").read()).hexdigest()
    nlines = sum(1 for _ in open(dst, encoding="utf-8"))
    print(f"{fname:12s} -> {dst}  ({nlines} lines)")
    print(f"             sha256 {digest}")

# Sanity: the modules import cleanly from the Drive folder.
import sys
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
import models, common
print()
print("import OK — CLASS_ORDER:", models.CLASS_ORDER)
print("BUILDERS:", list(models.BUILDERS))

models.py    -> /content/drive/MyDrive/bsm_paper_replication/models.py  (378 lines)
             sha256 ef94ca950cca677e74f26f6238ca8885b4a1ee182909197255c6fb5d6f6edcf4
common.py    -> /content/drive/MyDrive/bsm_paper_replication/common.py  (458 lines)
             sha256 36ec9a943969f5adb1b121d9f3c53b5772270722596a71a12f0bf7c60acb4816

import OK — CLASS_ORDER: ['Ilmenite', 'Rutile', 'Zircon', 'Garnet', 'Monazite', 'Sillimanite']
BUILDERS: ['dresnet50', 'resnet18', 'resnet34', 'vgg16']


## Verification: all four architectures on Colab (spec section 6)

Stage sizes 112/56/28/14/7, parameter counts vs the standard references, MACs vs the paper's FLOPs column. Must pass before any training notebook.

In [8]:
# ===== Verification: all four architectures (spec section 6) =====
import sys
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import torch
from models import MODEL_NAMES, format_verification, verify_architecture

results = []
for name in MODEL_NAMES:
    torch.manual_seed(42)
    results.append(verify_architecture(name))
print(format_verification(results))
ok = all(r["all_ok"] for r in results)
print()
print("ARCHITECTURE VERIFICATION (Colab, T4):", "PASSED" if ok else "FAILED")
assert ok, "architecture verification failed — do not proceed to training notebooks"

== dresnet50 ==
  stage sizes      : [112, 56, 28, 14, 7] (paper: [112, 56, 28, 14, 7]) -> OK
  output shape     : (2, 6) -> OK
  params (1000 hd) : 25,557,032 (ref 25,557,032) -> OK
  MACs             : 3.86G (paper FLOPs 3.8e+09) -> OK
  ALL OK           : True
== resnet18 ==
  stage sizes      : [112, 56, 28, 14, 7] (paper: [112, 56, 28, 14, 7]) -> OK
  output shape     : (2, 6) -> OK
  params (1000 hd) : 11,689,512 (ref 11,689,512) -> OK
  MACs             : 1.81G (paper FLOPs 1.8e+09) -> OK
  ALL OK           : True
== resnet34 ==
  stage sizes      : [112, 56, 28, 14, 7] (paper: [112, 56, 28, 14, 7]) -> OK
  output shape     : (2, 6) -> OK
  params (1000 hd) : 21,797,672 (ref 21,797,672) -> OK
  MACs             : 3.66G (paper FLOPs 3.6e+09) -> OK
  ALL OK           : True
== vgg16 ==
  stage sizes      : [112, 56, 28, 14, 7] (paper: [112, 56, 28, 14, 7]) -> OK
  output shape     : (2, 6) -> OK
  params (1000 hd) : 138,357,544 (ref 138,357,544) -> OK
  MACs             : 15.47G (

## Persistence: save this notebook to the project folder

Keeps a named copy of the executed notebook under the Drive project folder (spec section 3: local + Drive copies of every notebook).

In [9]:
# ===== Persistence: save executed notebook copy to the project folder =====
import glob, os, shutil, time

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
os.makedirs(os.path.join(PROJECT, "notebooks"), exist_ok=True)

# Colab autosaves this browser notebook into the account's "Colab Notebooks"
# folder; find the most recent .ipynb there and keep a named copy on the
# project folder.
candidates = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if not candidates:
    print("WARNING: no autosaved notebook found yet (Colab autosave can lag ~1 min).")
    print("Re-run this cell in a minute, or use File -> Save a copy in Drive.")
else:
    newest = max(candidates, key=os.path.getmtime)
    age_min = (time.time() - os.path.getmtime(newest)) / 60
    dst = os.path.join(PROJECT, "notebooks", "01_eda.ipynb")
    shutil.copy(newest, dst)
    print(f"copied: {newest}  (modified {age_min:.1f} min ago)")
    print(f"    -> {dst}")


Re-run this cell in a minute, or use File -> Save a copy in Drive.
